In [1]:
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

from src.load_model_data import get_patient_hba1c_cgm

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

In [2]:
patient_cgm = get_patient_hba1c_cgm()

In [3]:
def reduce_dims(X, method):
    if method == 'umap':
        reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=42)
    elif method == 'pca':
        reducer = PCA(n_components=3, random_state=42)
    elif method == 'tsne':
        reducer = TSNE(n_components=3, perplexity=30, random_state=42)
    else:
        raise ValueError(f"Unknown method: {method}")
    return reducer.fit_transform(X)

In [4]:
percentiles = np.arange(0, 101, 5)  # 21 features per patient
patient_ids = list(patient_cgm.keys())
hba1c_vals = np.array([patient_cgm[pid]['hba1c'] for pid in patient_ids])
X_patient = np.array([np.percentile(patient_cgm[pid]['cgm'], percentiles) for pid in patient_ids])
X_patient = StandardScaler().fit_transform(X_patient)

In [5]:
def plot_dim_red(emb, hba1c_vals, patient_ids, label):
    fig = go.Figure(go.Scatter3d(
        x=emb[:, 0], y=emb[:, 1], z=emb[:, 2],
        mode='markers',
        marker=dict(size=5, color=hba1c_vals, colorscale='Viridis',
                    colorbar=dict(title='HbA1c'), opacity=0.8),
        text=patient_ids,
        hovertemplate='%{text}<br>HbA1c: %{marker.color:.1f}<extra></extra>'
    ))

    fig.update_layout(
        title=f'{label} 3D of per-patient CGM percentiles, colored by HbA1c',
        scene=dict(xaxis_title=f'{label} 1', yaxis_title=f'{label} 2', zaxis_title=f'{label} 3'),
        height=700
    )
    fig.show()

In [7]:
METHOD = 'umap'  # 'umap', 'pca' or 'tsne'

emb = reduce_dims(X_patient, METHOD)
label = METHOD.upper()

plot_dim_red(emb=emb, hba1c_vals=hba1c_vals, patient_ids=patient_ids, label=label)

c:\Users\menzatiuko\vs_code_scripts\explore_diabetes_data\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [8]:
METHOD = 'pca'  # 'umap', 'pca' or 'tsne'

emb = reduce_dims(X_patient, METHOD)
label = METHOD.upper()

plot_dim_red(emb=emb, hba1c_vals=hba1c_vals, patient_ids=patient_ids, label=label)

looks like PCA1 can be a good feature extraction method as we can see some separation in colours in PC1 vs PC2 or PC1 vs PC3

In [9]:
METHOD = 'tsne'  # 'umap', 'pca' or 'tsne'

emb = reduce_dims(X_patient, METHOD)
label = METHOD.upper()

plot_dim_red(emb=emb, hba1c_vals=hba1c_vals, patient_ids=patient_ids, label=label)